In [ ]:
# libraries 
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns 

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping

In [ ]:
#data paths 
base_dir = "/kaggle/input/datasets/misrakahmed/vegetable-image-dataset/Vegetable Images"


In [ ]:
train_dir = os.path.join(base_dir, "train")
test_dir = os.path.join(base_dir, "test")
val_dir = os.path.join(base_dir, "validation")

In [ ]:
train_dir , test_dir , val_dir

In [ ]:
#count
def count_images(folder_path):
    counts = {}
    for class_name in os.listdir(folder_path):
        class_path = os.path.join(folder_path, class_name)
        if os.path.isdir(class_path):
            counts[class_name] = len(os.listdir(class_path))
    return counts

    

In [ ]:
train_counts = count_images(train_dir)
test_counts = count_images(test_dir)
val_counts = count_images(val_dir)

In [ ]:
train_counts

In [ ]:
test_counts

In [ ]:
val_counts

In [ ]:
#df
train_df = pd.DataFrame(list(train_counts.items()), columns=["Vegetable", "Train Count"])
test_df = pd.DataFrame(list(test_counts.items()), columns=["Vegetable", "Test Count"])
val_df = pd.DataFrame(list(val_counts.items()), columns=["Vegetable", "Validation Count"])

In [ ]:
summary_df = train_df.merge(test_df, on="Vegetable").merge(val_df, on="Vegetable")

In [ ]:
summary_df

In [ ]:
# visualization
summary_df.set_index("Vegetable").plot(kind="bar", figsize=(14, 7))

plt.title("Number of Images per Vegetable Class")
plt.xlabel("Vegetable")
plt.ylabel("Image Count")
plt.xticks(rotation=45)
plt.legend()
plt.show()


In [ ]:
# the data is balanced every class in every split 1000 images 800 train and 200 test -- all are the same 

In [ ]:
# img processing 
train_gen = ImageDataGenerator(
    rescale=1./255,rotation_range=20,width_shift_range=0.2,
    height_shift_range=0.2,shear_range=0.2,zoom_range=0.2,
    horizontal_flip=True,validation_split=0.2)

In [ ]:
val_gen = ImageDataGenerator(rescale=1./255,validation_split=0.2)

In [ ]:
# load images 
train_ds = train_gen.flow_from_directory(
    train_dir,target_size=(224, 224),batch_size=32,
    class_mode="categorical",subset="training",seed=42)

In [ ]:
val_ds = val_gen.flow_from_directory(
    train_dir,target_size=(224, 224),batch_size=32,
    class_mode="categorical",subset="validation",seed=42)

In [ ]:
train_ds.class_indices

In [ ]:
# cnn model 
model = Sequential([
    
    Conv2D(32, (3, 3), activation="relu", input_shape=(224, 224, 3)),
    MaxPooling2D((2, 2)),

    Conv2D(64, (3, 3), activation="relu"),
    MaxPooling2D((2, 2)),

    Conv2D(128, (3, 3), activation="relu"),
    MaxPooling2D((2, 2)),

    Flatten(),
    Dense(512, activation="relu"),
    Dropout(0.5),
    Dense(15, activation="softmax")
])

In [ ]:
# compile
model.compile(optimizer="adam",loss="categorical_crossentropy",metrics=["accuracy"])

In [ ]:
model.summary()

In [ ]:
## Conv layers: 3 (32, 64, 128 filters).
## Pooling: reduces the feature map size and keeps only the important features.
## Dropout: randomly turns off 50% of neurons to reduce the overfitting.
## Softmax: turns the outputs to probabilities for the 15 classes.

In [ ]:
# training model 

early_stop = EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)

history = model.fit(train_ds, validation_data=val_ds, epochs=20, callbacks=[early_stop])

In [ ]:
best_epoch = np.argmin(history.history["val_loss"]) + 1
print("Best epoch:", best_epoch)


In [ ]:
# Early stopping is good for saving time and prevent overfitting from the model to save the data during training 

In [ ]:
#evaluate the model 
train_loss, train_acc = model.evaluate(train_ds)
val_loss, val_acc = model.evaluate(val_ds)

In [ ]:
print(f'Training Accuracy: {round(train_acc, 4)}')
print(f'Validation Accuracy: {round(val_acc, 4)}')
print(f'Training Loss: {round(train_loss, 4)}')
print(f'Validation Loss: {round(val_loss, 4)}')

In [ ]:
# if the training accuracy is high and validation accuracy is low -  this means overfitting and the model memorizes the data 

In [ ]:
#plot acc / loss plot 
epochs_range = range(1, len(history.history["accuracy"]) + 1)

plt.figure(figsize=(12, 4))

# accuracy plot
plt.subplot(1, 2, 1)
plt.plot(epochs_range, history.history["accuracy"], label="Training Accuracy")
plt.plot(epochs_range, history.history["val_accuracy"], label="Validation Accuracy")
plt.title("Training vs Validation Accuracy")
plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.legend()

# loss plot
plt.subplot(1, 2, 2)
plt.plot(epochs_range, history.history["loss"], label="Training Loss")
plt.plot(epochs_range, history.history["val_loss"], label="Validation Loss")
plt.title("Training vs Validation Loss")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.legend()

plt.show()
